In [1]:
import sys
# Define the path to the directory containing the module
module_dir = "../Main/RequiredFuntions"

# Append the directory to sys.path
sys.path.append(module_dir)

In [2]:
import time
import json
import hashlib
import threading
import functions as fn
import dataTransfer as DT
import EncryptionDecryption as ED

In [ ]:
with open('../Main/ReceivedData/keys.json', 'r') as file:
    keys = json.load(file)

with open('../Main/ReceivedData/datastore.json', 'r') as file:
    user = json.load(file)

with open('../Main/ReceivedData/datastore_device.json', 'r') as file:
    device = json.load(file)

In [ ]:
presharedkeys = keys['key']

In [5]:
userNonce_i = fn.nonce_gen()

In [6]:
uniqueId_i = "RajeshDevice1"
uniqueId_g = "rajeshGateway"
uniqueId_j = "RajeshHomeDevice1"

In [7]:
temporalIdentity_i = hashlib.sha256((uniqueId_i + keys['user_public'] + str(userNonce_i)).encode()).hexdigest()
temporalIdentity_g = hashlib.sha256((uniqueId_g + keys['gateway_public'] + str(userNonce_i)).encode()).hexdigest()
temporalIdentity_j = hashlib.sha256((uniqueId_j + keys['device_public'] + str(userNonce_i)).encode()).hexdigest()

In [8]:
computeI = hashlib.sha256((temporalIdentity_i + temporalIdentity_g + temporalIdentity_j + str(userNonce_i)).encode()).hexdigest()

In [9]:
# step 1
fingerprintImagePath = "../Main/RequiredData/Fingerprint/11.jpg"
accelerometerDataPath = "../Main/RequiredData/Accelerometer/testingdataM55.csv"

In [10]:
imageHash = fn.hash_file(fingerprintImagePath)

accelerometerHash = fn.hash_file(accelerometerDataPath)

In [11]:
encryptedIdandImage = ED.symmetric_key_encryption('', 
                                                  ED.read_image_as_bytes(fingerprintImagePath), 
                                                  presharedkeys)

encryptedIDandAccelerometerData = ED.symmetric_key_encryption('',
                                                              ED.read_image_as_bytes(accelerometerDataPath), 
                                                              presharedkeys)

In [12]:
userDeviceData = {
    "encryptedImage" : encryptedIdandImage,
    "encryptedAccelerometerData" : encryptedIDandAccelerometerData,
    "imageHash" : imageHash,
    "accelerometerHash" : accelerometerHash,
    "userNonce" : userNonce_i,
    "compute" : computeI,
    "TDi" : temporalIdentity_i,
    "TDj" : temporalIdentity_j,
    "TDg" : temporalIdentity_g
}

In [13]:
# Create and start threads
thread1 = threading.Thread(target=DT.receive,  args=("keyGeneration/authentication_user_send.json",))  # Start the receive function
thread2 = threading.Thread(target=DT.send, args=(userDeviceData,))  # Start the send function with arguments

thread1.start()
time.sleep(2)  # Ensure the server starts before sending
thread2.start()

# Wait for threads to complete
thread1.join()
thread2.join()

print("Data transfer and storage complete.")

Server is listening on port 12345...
Connected by ('127.0.0.1', 64173)
Data transfer done.
Received data saved to 'received_data.json'.
Data transfer and storage complete.


In [14]:
# step 2
print("Starting Step 2")
json_data = fn.read_json_file("../Main/ReceivedData/keyGeneration/authentication_user_send.json")

Starting Step 2


In [15]:
Md = device[uniqueId_g][uniqueId_j]["partialSecretIntegrityUser"]
Mu = user[uniqueId_g][uniqueId_i]["partialSecretIntegrityUser"]

In [16]:
gatewayNonce = fn.nonce_gen()

In [17]:
M = fn.xor_strings(Mu, Md)
M = fn.xor_strings(M, str(gatewayNonce))

In [18]:
Pu = fn.xor_strings(str(gatewayNonce), str(user[uniqueId_g][uniqueId_i]["gatewayShare"]))
Pu = fn.xor_strings(Pu, uniqueId_i)

In [19]:
Pd = fn.xor_strings(str(gatewayNonce), str(device[uniqueId_g][uniqueId_j]["gatewayShare"]))
Pd = fn.xor_strings(Pd, uniqueId_j)

In [20]:
gatewayNonce_device = fn.nonce_gen()
gatewayNonce_user = fn.nonce_gen()

In [21]:
gatewayComputeJ = fn.xor_strings(hashlib.sha256(
    (
        M + Pd + str(gatewayNonce_device) + uniqueId_j
    ).encode()
).hexdigest(), str(device[uniqueId_g][uniqueId_j]["gatewayShare"]))

gatewayComputeI = fn.xor_strings(hashlib.sha256(
    (
        M + Pu + str(gatewayNonce_user) + uniqueId_i
    ).encode()
).hexdigest(), str(user[uniqueId_g][uniqueId_i]["gatewayShare"]))

In [22]:
deviceAuthenticationMessage = {
    "M" : M,
    "Pd" : Pd,
    "gatewayNonce_device": gatewayNonce_device,
    "gatewayComputeJ": gatewayComputeJ
}

In [23]:
extractedDeviceSharesJ = fn.xor_strings(deviceAuthenticationMessage["gatewayComputeJ"] ,hashlib.sha256(
    (
        deviceAuthenticationMessage["M"] + deviceAuthenticationMessage["Pd"] + str(deviceAuthenticationMessage["gatewayNonce_device"]) + uniqueId_j
    ).encode()
).hexdigest())

In [30]:
hashlib.sha256(extractedDeviceSharesJ.encode()).hexdigest()

'e7f6c011776e8db7cd330b54174fd76f7d0216b612387a5ffcfb81e6f0919683'

In [28]:
hashlib.sha256(
    (
        deviceAuthenticationMessage["M"] + deviceAuthenticationMessage["Pd"] + str(deviceAuthenticationMessage["gatewayNonce_device"]) + uniqueId_j
    ).encode()
).hexdigest()

'e9a0213315997ba1e9be9ae57b0bfa0b7e7e266e32960484a37dd032113ab184'

In [24]:
reconstructedSharesJ = (2 * int(extractedDeviceSharesJ) - device[uniqueId_g][uniqueId_j]["userShare"]) % fn.primeNumbergenerator()

In [25]:
reconstructedSharesJ

0

In [26]:
hashlib.sha256(
    (
        str(reconstructedSharesJ) + uniqueId_g
    ).encode()
).hexdigest()

'9c6472440241b6eb3d897316bbbc3968ce27dd35b4f938a58b83c5787c5ae506'